In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 6 Commit 1: Prepare Tableau Data for Dashboard 1 & Dashboard 2
# feat(task6): export mortgage overview and model evaluation datasets
#
# Coursework Requirement:
# Prepare Tableau-ready datasets for visual analytics.
# Dashboard 1 : Mortgage Application Overview
# Dashboard 2 : Machine Learning Model Performance
# ──────────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import pandas as pd
import builtins

os.environ["JAVA_HOME"]="/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"]=sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("7006SCN_HMDA_Task6")
    .master("local[*]")
    .config("spark.driver.memory","10g")
    .config("spark.sql.shuffle.partitions","200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# ------------------------------------------------------------------------------
# Project Paths
# ------------------------------------------------------------------------------

BASE="/content/drive/MyDrive/7006SCN"

PROCESSED=f"{BASE}/data/processed"

OUTPUT=f"{BASE}/outputs"

print("="*80)
print("TASK 6 : TABLEAU DATA PREPARATION")
print("="*80)

# ------------------------------------------------------------------------------
# Load Clean HMDA Dataset
# ------------------------------------------------------------------------------

df=spark.read.parquet(
    f"{PROCESSED}/hmda_clean.parquet"
)

print(f"Rows    : {df.count():,}")
print(f"Columns : {len(df.columns)}")

# ==============================================================================
# DASHBOARD 1
# Mortgage Application Overview
# ==============================================================================

print("\nPreparing Dashboard 1...")

# ------------------------------------------------------------------------------
# KPI Summary
# ------------------------------------------------------------------------------

total=df.count()

approved=df.filter(
    col("loan_approval")==1
).count()

rejected=df.filter(
    col("loan_approval")==0
).count()

approval_rate = builtins.round(
    approved / total * 100,
    2
)

kpi=pd.DataFrame({

    "Metric":[
        "Total Applications",
        "Approved Loans",
        "Rejected Loans",
        "Approval Rate (%)"
    ],

    "Value":[
        total,
        approved,
        rejected,
        approval_rate
    ]

})

kpi.to_csv(
    f"{OUTPUT}/t6_dashboard1_kpi.csv",
    index=False
)

# ------------------------------------------------------------------------------
# Missing Values
# ------------------------------------------------------------------------------

missing=[]

for c in df.columns:

    missing.append({

        "Column":c,

        "Missing Values":df.filter(
            col(c).isNull()
        ).count()

    })

missing_df=pd.DataFrame(missing)

missing_df.to_csv(

    f"{OUTPUT}/t6_dashboard1_missing.csv",

    index=False

)

# ------------------------------------------------------------------------------
# Loan Approval Distribution
# ------------------------------------------------------------------------------

approval=(
    df.groupBy("loan_approval")
      .count()
      .toPandas()
)

approval.to_csv(

    f"{OUTPUT}/t6_dashboard1_approval.csv",

    index=False

)

# ------------------------------------------------------------------------------
# Loan Purpose Distribution
# ------------------------------------------------------------------------------

if "loan_purpose" in df.columns:

    purpose=(
        df.groupBy("loan_purpose")
          .count()
          .toPandas()
    )

    purpose.to_csv(

        f"{OUTPUT}/t6_dashboard1_loanpurpose.csv",

        index=False

    )

# ------------------------------------------------------------------------------
# Race Distribution
# ------------------------------------------------------------------------------

if "applicant_race_1" in df.columns:

    race=(
        df.groupBy("applicant_race_1")
          .count()
          .toPandas()
    )

    race.to_csv(

        f"{OUTPUT}/t6_dashboard1_race.csv",

        index=False

    )

# ------------------------------------------------------------------------------
# Ethnicity Distribution
# ------------------------------------------------------------------------------

if "applicant_ethnicity_1" in df.columns:

    ethnicity=(
        df.groupBy("applicant_ethnicity_1")
          .count()
          .toPandas()
    )

    ethnicity.to_csv(

        f"{OUTPUT}/t6_dashboard1_ethnicity.csv",

        index=False

    )

print("Dashboard 1 datasets exported ✓")

# ==============================================================================
# DASHBOARD 2
# Machine Learning Performance
# ==============================================================================

print("\nPreparing Dashboard 2...")

metrics=pd.read_csv(
    f"{OUTPUT}/task5_metrics.csv"
)

metrics.to_csv(

    f"{OUTPUT}/t6_dashboard2_model_metrics.csv",

    index=False

)

print("Model metrics loaded ✓")

# ------------------------------------------------------------------------------
# SHAP Feature Importance
# ------------------------------------------------------------------------------

shap_df=pd.read_csv(

    f"{OUTPUT}/task5_shap.csv"

)

shap_df.to_csv(

    f"{OUTPUT}/t6_dashboard2_shap.csv",

    index=False

)

print("SHAP importance loaded ✓")

# ------------------------------------------------------------------------------
# Prediction Time Dataset
# ------------------------------------------------------------------------------

prediction=metrics[

    [

        "Model",

        "Prediction Time (s)"

    ]

]

prediction.to_csv(

    f"{OUTPUT}/t6_dashboard2_prediction_time.csv",

    index=False

)

# ------------------------------------------------------------------------------
# ROC Comparison
# ------------------------------------------------------------------------------

roc=metrics[

    [

        "Model",

        "ROC-AUC"

    ]

]

roc.to_csv(

    f"{OUTPUT}/t6_dashboard2_roc.csv",

    index=False

)

print("Dashboard 2 datasets exported ✓")

print()

print("="*80)
print("COMMIT 1 COMPLETED")
print("="*80)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Commit 2: Prepare Tableau Data for Dashboard 3 & Dashboard 4
# feat(task6): export business intelligence and Spark performance datasets
#
# Coursework Requirement:
# Prepare Tableau-ready datasets for advanced visual analytics.
# Dashboard 3 : Mortgage Business Intelligence
# Dashboard 4 : Spark Performance Analytics
# ──────────────────────────────────────────────────────────────────────────────

from pyspark.sql.functions import *
import pandas as pd

print("="*80)
print("COMMIT 2 : DASHBOARD 3 & DASHBOARD 4")
print("="*80)

# =============================================================================
# DASHBOARD 3
# Mortgage Business Intelligence
# =============================================================================

print("\nPreparing Dashboard 3...")

# -----------------------------------------------------------------------------
# Approval Rate by State
# -----------------------------------------------------------------------------

if "state_code" in df.columns:

    state = (
        df.groupBy("state_code")
          .agg(
              count("*").alias("Applications"),
              sum(col("loan_approval")).alias("Approved")
          )
          .withColumn(
              "Approval Rate (%)",
              round(col("Approved")/col("Applications")*100,2)
          )
          .toPandas()
    )

    state.to_csv(
        f"{OUTPUT}/t6_dashboard3_state.csv",
        index=False
    )

# -----------------------------------------------------------------------------
# Loan Purpose
# -----------------------------------------------------------------------------

if "loan_purpose" in df.columns:

    purpose = (
        df.groupBy(
            "loan_purpose",
            "loan_approval"
        )
        .count()
        .toPandas()
    )

    purpose.to_csv(
        f"{OUTPUT}/t6_dashboard3_loanpurpose.csv",
        index=False
    )

# -----------------------------------------------------------------------------
# Income Bands
# -----------------------------------------------------------------------------

if "income_num" in df.columns:

    income = (
        df.withColumn(
            "Income Band",
            when(col("income_num") < 50000,"<50K")
            .when(col("income_num") < 100000,"50K-100K")
            .when(col("income_num") < 150000,"100K-150K")
            .otherwise(">150K")
        )
        .groupBy(
            "Income Band",
            "loan_approval"
        )
        .count()
        .toPandas()
    )

    income.to_csv(
        f"{OUTPUT}/t6_dashboard3_income.csv",
        index=False
    )

# -----------------------------------------------------------------------------
# Loan Amount Distribution
# -----------------------------------------------------------------------------

if "loan_amount_num" in df.columns:

    loan = (
        df.select("loan_amount_num")
          .toPandas()
    )

    loan.to_csv(
        f"{OUTPUT}/t6_dashboard3_loanamount.csv",
        index=False
    )

# -----------------------------------------------------------------------------
# Interest Rate Distribution
# -----------------------------------------------------------------------------

if "interest_rate_num" in df.columns:

    interest = (
        df.select("interest_rate_num")
          .toPandas()
    )

    interest.to_csv(
        f"{OUTPUT}/t6_dashboard3_interest.csv",
        index=False
    )

# -----------------------------------------------------------------------------
# Race × Approval
# -----------------------------------------------------------------------------

if "applicant_race_1" in df.columns:

    race = (
        df.groupBy(
            "applicant_race_1",
            "loan_approval"
        )
        .count()
        .toPandas()
    )

    race.to_csv(
        f"{OUTPUT}/t6_dashboard3_raceapproval.csv",
        index=False
    )

# -----------------------------------------------------------------------------
# Ethnicity × Approval
# -----------------------------------------------------------------------------

if "applicant_ethnicity_1" in df.columns:

    ethnicity = (
        df.groupBy(
            "applicant_ethnicity_1",
            "loan_approval"
        )
        .count()
        .toPandas()
    )

    ethnicity.to_csv(
        f"{OUTPUT}/t6_dashboard3_ethnicityapproval.csv",
        index=False
    )

print("Dashboard 3 datasets exported ✓")

# =============================================================================
# DASHBOARD 4
# Spark Performance Analytics
# =============================================================================

print("\nPreparing Dashboard 4...")

# -----------------------------------------------------------------------------
# Model Runtime
# -----------------------------------------------------------------------------

metrics = pd.read_csv(
    f"{OUTPUT}/task5_metrics.csv"
)

runtime = metrics[[
    "Model",
    "Prediction Time (s)"
]]

runtime.to_csv(
    f"{OUTPUT}/t6_dashboard4_runtime.csv",
    index=False
)

# -----------------------------------------------------------------------------
# Spark Configuration
# -----------------------------------------------------------------------------

config = pd.DataFrame({

    "Configuration":[
        "Driver Memory",
        "Shuffle Partitions",
        "Spark Version"
    ],

    "Value":[
        spark.conf.get("spark.driver.memory"),
        spark.conf.get("spark.sql.shuffle.partitions"),
        spark.version
    ]

})

config.to_csv(
    f"{OUTPUT}/t6_dashboard4_config.csv",
    index=False
)

# -----------------------------------------------------------------------------
# Cache Benchmark
# -----------------------------------------------------------------------------

cache = pd.DataFrame({

    "Execution":[
        "Cold",
        "Warm"
    ],

    "Time (Seconds)":[
        7.03,
        30.77
    ]

})

cache.to_csv(
    f"{OUTPUT}/t6_dashboard4_cache.csv",
    index=False
)

# -----------------------------------------------------------------------------
# Partition Statistics
# -----------------------------------------------------------------------------

partition = pd.DataFrame({

    "Metric":[
        "Original Partitions",
        "Repartitioned Dataset",
        "Final Partitions",
        "Skew Ratio"
    ],

    "Value":[
        200,
        200,
        128,
        1.92
    ]

})

partition.to_csv(
    f"{OUTPUT}/t6_dashboard4_partition.csv",
    index=False
)

print("Dashboard 4 datasets exported ✓")

print()

print("="*80)
print("COMMIT 2 COMPLETED")
print("="*80)